# Trading H4+H1+M15 — v4 (Colab, cella unica)
**Scanner live + Backtest v3 vs v4** — output in `Il tuo Drive/TRADING_V4/`

### Istruzioni
1. API key Twelvedata nei **Colab Secrets** come `TWELVEDATA_API_KEY` (già fatto per la v3).
2. Nella cella di codice, riga `MODE = "scan"`:
   - `"backtest"` → **da eseguire per primo**: confronta le regole v3 e v4 sugli stessi dati storici (~15 strumenti × 3 chiamate ≈ 6–8 min).
   - `"scan"` → cerca i segnali di adesso (≈ 3 chiamate per strumento aperto, ~15 min).
3. Premi ▶ una volta e autorizza Google Drive al primo avvio.
4. (Facoltativo) Crea `TRADING_V4/posizioni_aperte.csv` con colonne `Nome prodotto,Direzione` per attivare il controllo di esposizione.

> I parametri v4 sono ipotesi da validare: usa lo scanner solo dopo che il backtest ha dato expectancy positiva su almeno 100 trade.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
===============================================================================
 TRADING H4 + H1 + M15 — v4  (scanner live + backtest v3 vs v4)
===============================================================================
 Cosa cambia rispetto alla v3 (vedi CHANGELOG in fondo al file per i dettagli):
   1. BUG Fibonacci: la tolleranza era il 5% del prezzo -> la zona risultava
      SEMPRE "38.2%". Ora la zona è calcolata con tolleranza in ATR.
   2. BUG candela incompleta: la v3 usava anche la candela ancora in formazione
      (una "pinbar" poteva sparire dopo pochi minuti). Ora si usano solo
      candele CHIUSE.
   3. Stop: non più "0,1% sotto la pinbar" ma basato sulla volatilità
      (minimo 1 x ATR H4), size calcolata per rischiare sempre la stessa cifra.
   4. Ingresso solo nella zona 50–61,8% (obbligatoria, non più bonus).
   5. Target prima del massimo/minimo precedente: se non c'è spazio per
      almeno 2R il trade viene scartato (la v3 metteva sempre TP = 2,5R).
   6. Filtro weekend (niente nuovi ingressi da venerdì 18:00 a domenica) e
      filtro esposizione (max 1 posizione per valuta nella stessa direzione).
   7. Etichette S1–S6 eliminate: al loro posto GATE obbligatori + BONUS 0–3
      con la STESSA definizione nel prompt e nel codice.
   8. Nuova modalità BACKTEST: riproduce le regole v3 e v4 sugli stessi dati
      storici e confronta i risultati in R (multipli del rischio).

 USO
   Colab: incolla il file in una cella (o usa il notebook fornito), imposta
          MODE qui sotto e premi ▶.
   Locale: pip install requests pandas numpy openpyxl pytz
           python trading_h4h1m15_v4.py            -> MODE di default
           python trading_h4h1m15_v4.py backtest   -> forza backtest
           python trading_h4h1m15_v4.py scan       -> forza scanner

 AVVERTENZA: strumento didattico. Nessun parametro v4 è ancora validato:
 il backtest serve proprio a verificarlo prima di usarlo, anche in demo.
===============================================================================
"""

import os, sys, time, math, re, warnings
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURAZIONE
# =============================================================================
MODE = "scan"            # "scan" = segnali di adesso | "backtest" = confronto v3 vs v4
for _a in sys.argv[1:]:
    if _a in ("scan", "backtest"):
        MODE = _a

RISK_EUR       = 10.0    # € che accetti di perdere se scatta lo stop (size piena)
REQUEST_DELAY  = 8.0     # secondi tra chiamate API (piano free Twelvedata: 8 req/min)
BACKTEST_PRESETS = ["v3", "v4", "v4_weekend_flat"]   # varianti confrontate nel backtest

# Parametri della strategia. Modificali SOLO dopo averli verificati col backtest.
PRESETS = {
    "v4": dict(
        ema_slope_bars   = 6,     # EMA50 H4 deve salire (long) rispetto a 6 candele H4 prima
        impulse_lookback = 60,    # candele H4 in cui cercare l'ultimo impulso
        min_impulse_atr  = 3.0,   # impulso minimo = 3 x ATR H4 (swing "vero", non rumore)
        fib_tol_atr      = 0.25,  # tolleranza zona 50–61,8% = 0,25 x ATR H4
        level_lookback   = 60,    # candele H1 in cui cercare supporti/resistenze
        level_tol_atr    = 0.30,  # due minimi H1 sono "stesso livello" se distano < 0,3 ATR H1
        min_touches      = 2,
        rsi_h1_long      = (30, 60),   # H1 né in caduta libera né già ripartito
        rsi_h1_short     = (40, 70),
        rsi_m15_long     = (30, 55),   # RSI M15 in risalita dentro questa fascia
        rsi_m15_short    = (45, 70),
        pin_wick         = 0.60,  # mèche >= 60% del range della candela
        pin_body         = 0.35,  # corpo <= 35% del range
        pin_min_atr      = 0.8,   # range pinbar >= 0,8 x ATR M15 (no candele insignificanti)
        pin_lookback     = 3,     # scanner: pinbar tra le ultime 3 M15 chiuse (backtest: 1)
        max_chase_R      = 0.30,  # se il prezzo si è già allontanato > 0,3R dalla pinbar: niente ingresso
        stop_buffer_atr1 = 0.25,  # stop oltre pinbar/livello di 0,25 x ATR H1
        min_stop_atr4    = 1.0,   # stop mai più stretto di 1 x ATR H4
        rr_target        = 2.5,
        rr_min           = 2.0,   # se prima del massimo precedente non c'è spazio per 2R -> scarta
        max_spread_R     = 0.10,  # spread stimato <= 10% del rischio
        time_stop_hours  = 240,   # chiusura forzata dopo 10 giorni di calendario
        close_before_weekend = False,
        weekend_filter   = True,
        exposure_filter  = True,
    ),
}
PRESETS["v4_weekend_flat"] = dict(PRESETS["v4"], close_before_weekend=True)
# "v3" non usa questi parametri: è la riproduzione fedele della logica precedente.

# Esposizione: quante posizioni nella stessa direzione per "chiave"
EXPOSURE_LIMIT = {"EQ_US": 2, "EQ_EU": 1, "EQ_ASIA": 1}   # default per valute = 1

# Spread stimati in % del prezzo  [STIME DA VERIFICARE sulla tua piattaforma]
SPREAD_PCT = {"fx_major": 0.012, "fx_cross": 0.025, "metal": 0.035,
              "index": 0.015, "stock_us": 0.10, "stock_eu": 0.12, "stock_asia": 0.15}

# ---- Strumenti (piano free Twelvedata, lista v3) ------------------------------
# nome: (simbolo TD, exchange, classe, mercato)
FX_MAJ = ["EUR/USD", "USD/JPY", "GBP/USD", "USD/CHF", "AUD/USD"]
FX_CRS = ["EUR/GBP", "EUR/CHF", "GBP/CHF", "EUR/CAD", "EUR/AUD", "EUR/JPY", "CAD/JPY", "CHF/JPY"]
INSTRUMENTS = {p: (p, None, "fx_major", "fx") for p in FX_MAJ}
INSTRUMENTS.update({p: (p, None, "fx_cross", "fx") for p in FX_CRS})
INSTRUMENTS.update({
    "Oro (XAU/USD)":         ("XAU/USD", None,     "metal",    "fx"),
    "DAX 40":                ("DAX",     None,     "index",    "europe"),
    "Apple (AAPL)":          ("AAPL",  "NASDAQ",   "stock_us", "us"),
    "Microsoft (MSFT)":      ("MSFT",  "NASDAQ",   "stock_us", "us"),
    "NVIDIA (NVDA)":         ("NVDA",  "NASDAQ",   "stock_us", "us"),
    "Tesla (TSLA)":          ("TSLA",  "NASDAQ",   "stock_us", "us"),
    "Amazon (AMZN)":         ("AMZN",  "NASDAQ",   "stock_us", "us"),
    "Meta (META)":           ("META",  "NASDAQ",   "stock_us", "us"),
    "Alphabet (GOOGL)":      ("GOOGL", "NASDAQ",   "stock_us", "us"),
    "Bank of America (BAC)": ("BAC",   "NYSE",     "stock_us", "us"),
    "Intel (INTC)":          ("INTC",  "NASDAQ",   "stock_us", "us"),
    "JPMorgan (JPM)":        ("JPM",   "NYSE",     "stock_us", "us"),
    "Visa (V)":              ("V",     "NYSE",     "stock_us", "us"),
    "Mastercard (MA)":       ("MA",    "NYSE",     "stock_us", "us"),
    "P&G (PG)":              ("PG",    "NYSE",     "stock_us", "us"),
    "Coca-Cola (KO)":        ("KO",    "NYSE",     "stock_us", "us"),
    "Berkshire B (BRK.B)":   ("BRK.B", "NYSE",     "stock_us", "us"),
    "ASML":                  ("ASML",  "NASDAQ",   "stock_us", "us"),   # ADR su NASDAQ
    "SAP":                   ("SAP",   "NYSE",     "stock_us", "us"),   # ADR su NYSE
    # Asia: nella v3 risultavano "da verificare" sul piano free -> disattivate.
    # "Toyota (7203)": ("7203", "TSE", "stock_asia", "tokyo"),
})
# Strumenti usati nel backtest (3 chiamate ciascuno). Tieni la lista corta.
BACKTEST_SYMBOLS = FX_MAJ + FX_CRS + ["Oro (XAU/USD)"]

# Sessioni (ora di Roma). Forex/oro: 24h dal lunedì al venerdì.
SESSIONS = {"tokyo": (2, 9), "europe": (9, 17.5), "us": (15.5, 22)}

BASE_URL = "https://api.twelvedata.com/time_series"
INTERVAL_MIN = {"4h": 240, "1h": 60, "15min": 15}

# =============================================================================
# AMBIENTE (Colab / locale) + API KEY
# =============================================================================
try:
    import pytz
    ROME = pytz.timezone("Europe/Rome")
except ImportError:
    from zoneinfo import ZoneInfo
    ROME = ZoneInfo("Europe/Rome")

try:
    from google.colab import drive, userdata   # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def get_api_key():
    k = os.environ.get("TWELVEDATA_API_KEY", "")
    if not k and IN_COLAB:
        try:
            k = userdata.get("TWELVEDATA_API_KEY")
        except Exception:
            k = ""
    return k or ""

def get_out_dir():
    if IN_COLAB:
        drive.mount("/content/drive", force_remount=False)
        d = "/content/drive/MyDrive/TRADING_V4"
    else:
        d = os.path.join(os.getcwd(), "TRADING_V4")
    os.makedirs(os.path.join(d, "cache"), exist_ok=True)
    return d

def log(m=""):
    print(m, flush=True)

# =============================================================================
# DATI
# =============================================================================
def fetch_td(symbol, exchange, interval, n, api_key):
    """Scarica candele da Twelvedata in UTC. Restituisce (df, messaggio)."""
    params = {"symbol": symbol, "interval": interval, "outputsize": n, "apikey": api_key,
              "format": "JSON", "order": "ASC", "timezone": "UTC"}
    if exchange:
        params["exchange"] = exchange
    for attempt in range(2):
        try:
            j = requests.get(BASE_URL, params=params, timeout=30).json()
        except Exception as e:
            return None, f"RETE: {e}"
        if j.get("status") == "error" or "values" not in j:
            code = j.get("code", "?")
            if code == 429 and attempt == 0:
                log("    [RATE LIMIT] attendo 65s..."); time.sleep(65); continue
            label = {400: "SIMBOLO NON TROVATO", 401: "API KEY NON VALIDA",
                     403: "PIANO NON SUPPORTATO", 404: "DATI NON DISPONIBILI"}.get(code, f"ERR {code}")
            return None, f"{label}: {str(j.get('message', ''))[:80]}"
        df = pd.DataFrame(j["values"])
        df["time"] = pd.to_datetime(df["datetime"], utc=True)
        for c in ("open", "high", "low", "close"):
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df = df[["time", "open", "high", "low", "close"]].dropna()
        df = df.sort_values("time").drop_duplicates("time").reset_index(drop=True)
        return df, "OK"
    return None, "RETRY ESAURITO"

def drop_incomplete(df, interval, now_utc):
    """FIX v4: elimina la candela ancora in formazione."""
    if df is None or df.empty:
        return df
    close_t = df["time"] + pd.Timedelta(minutes=INTERVAL_MIN[interval])
    return df[close_t <= now_utc].reset_index(drop=True)

def validate(df, min_rows):
    if df is None or len(df) < min_rows:
        return False
    bad = ((df.high < df.low) | (df.close > df.high) | (df.close < df.low) |
           (df.open > df.high) | (df.open < df.low) | (df.close <= 0))
    return not bad.any()

# =============================================================================
# INDICATORI (tutti causali: il valore alla candela k usa solo dati <= k)
# =============================================================================
def rsi_wilder(close, period=14):
    d = close.diff()
    up = d.clip(lower=0).ewm(alpha=1 / period, adjust=False).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1 / period, adjust=False).mean()
    rs = up / dn.replace(0, np.nan)
    return (100 - 100 / (1 + rs)).fillna(100)

def atr(df, period=14):
    pc = df.close.shift(1)
    tr = pd.concat([df.high - df.low, (df.high - pc).abs(), (df.low - pc).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / period, adjust=False).mean()

def add_features(df, tf):
    df = df.copy()
    if tf == "4h":
        df["ema50"] = df.close.ewm(span=50, adjust=False).mean()
        df["ema200"] = df.close.ewm(span=200, adjust=False).mean()
    elif tf == "1h":
        df["ema20"] = df.close.ewm(span=20, adjust=False).mean()
        df["ema50"] = df.close.ewm(span=50, adjust=False).mean()
        df["rsi"] = rsi_wilder(df.close)
    else:
        df["ema20"] = df.close.ewm(span=20, adjust=False).mean()
        df["rsi"] = rsi_wilder(df.close)
    df["atr"] = atr(df)
    return df

def fractals(high, low, k=2):
    """Swing high/low confermati (k candele per lato). Restituisce due liste di indici."""
    n = len(high); sh, sl = [], []
    for i in range(k, n - k):
        if high[i] >= high[i - k:i].max() and high[i] >= high[i + 1:i + k + 1].max():
            sh.append(i)
        if low[i] <= low[i - k:i].min() and low[i] <= low[i + 1:i + k + 1].min():
            sl.append(i)
    return sh, sl

# =============================================================================
# ESPOSIZIONE PER VALUTA / GRUPPO
# =============================================================================
def exposure_keys(name, direction):
    """Es. EUR/CHF buy -> {EUR:+1, CHF:-1}; Apple buy -> {EQ_US:+1}."""
    sym, _, cls, mkt = INSTRUMENTS.get(name, (name, None, "fx_cross", "fx"))
    s = 1 if direction == "buy" else -1
    if "/" in sym:
        a, b = sym.split("/")
        return {a: s, b: -s}
    if cls == "index":
        return {"EQ_EU" if mkt == "europe" else "EQ_US": s}
    return {"EQ_US" if cls == "stock_us" else ("EQ_EU" if cls == "stock_eu" else "EQ_ASIA"): s}

def exposure_ok(name, direction, open_list):
    """open_list: lista di (nome, direzione) già aperti."""
    counts = {}
    for n, d in open_list:
        for k, v in exposure_keys(n, d).items():
            counts[(k, v)] = counts.get((k, v), 0) + 1
    for k, v in exposure_keys(name, direction).items():
        if counts.get((k, v), 0) >= EXPOSURE_LIMIT.get(k, 1):
            return False, f"già esposto su {k} {'long' if v > 0 else 'short'}"
    return True, ""

# =============================================================================
# SESSIONI / WEEKEND
# =============================================================================
def market_open(name, now_rome):
    _, _, cls, mkt = INSTRUMENTS[name]
    wd, h = now_rome.weekday(), now_rome.hour + now_rome.minute / 60
    if mkt == "fx":
        if wd == 5 or (wd == 6 and h < 23) or (wd == 4 and h >= 23):
            return False
        return True
    if wd >= 5:
        return False
    s, e = SESSIONS[mkt]
    return s <= h < e

def weekend_block(now_rome):
    """FIX v4: niente nuovi ingressi da venerdì 18:00 a domenica."""
    wd, h = now_rome.weekday(), now_rome.hour
    return (wd == 4 and h >= 18) or wd >= 5

# =============================================================================
# LOGICA v4
# =============================================================================
def evaluate_v4(h4, h1, m15, P, name, now_utc=None):
    """
    h4/h1/m15: DataFrame con feature e SOLO candele chiuse (ultima riga = ultima chiusa).
    Ritorna dict con 'ok' (bool), 'gate' (motivo di scarto o 'TUTTI I GATE'), e i dettagli.
    """
    r = {"ok": False, "gate": "", "bonus": 0, "bonus_det": []}
    cls = INSTRUMENTS[name][2]
    a4 = float(h4.atr.iloc[-1]); a1 = float(h1.atr.iloc[-1]); a15 = float(m15.atr.iloc[-1])
    last4 = h4.iloc[-1]; price = float(m15.close.iloc[-1])
    r.update({"ATR H4": a4, "EMA H4": "", "RSI H1": round(float(h1.rsi.iloc[-1]), 1)})

    # G1 — bias H4: EMA50 vs EMA200 + pendenza EMA50 + prezzo dal lato giusto della EMA200
    e50, e200 = float(last4.ema50), float(last4.ema200)
    e50_prev = float(h4.ema50.iloc[-1 - P["ema_slope_bars"]])
    if e50 > e200 and e50 > e50_prev and last4.close > e200:
        d = "buy"; r["EMA H4"] = "long"
    elif e50 < e200 and e50 < e50_prev and last4.close < e200:
        d = "sell"; r["EMA H4"] = "short"
    else:
        r["gate"] = "G1 BIAS H4 NON DEFINITO"; r["EMA H4"] = "mista"; return r
    r["Direzione"] = d
    s = 1 if d == "buy" else -1

    # G2 — impulso H4 e zona Fibonacci 50–61,8%
    w = h4.iloc[-P["impulse_lookback"]:]
    hi, lo = w.high.values, w.low.values
    sh, sl = fractals(hi, lo)
    if d == "buy":
        if not sh: r["gate"] = "G2 SWING H4 NON IDENTIFICABILE"; return r
        ih = sh[-1]; ext = hi[ih]
        if ih < 3: r["gate"] = "G2 SWING H4 NON IDENTIFICABILE"; return r
        start = lo[max(0, ih - 30):ih].min()
        if hi[ih + 1:].max(initial=-np.inf) > ext or lo[ih + 1:].min(initial=np.inf) <= start:
            r["gate"] = "G2 IMPULSO H4 NON VALIDO (superato)"; return r
        rng = ext - start
        z50, z618 = ext - 0.5 * rng, ext - 0.618 * rng
    else:
        if not sl: r["gate"] = "G2 SWING H4 NON IDENTIFICABILE"; return r
        il = sl[-1]; ext = lo[il]
        if il < 3: r["gate"] = "G2 SWING H4 NON IDENTIFICABILE"; return r
        start = hi[max(0, il - 30):il].max()
        if lo[il + 1:].min(initial=np.inf) < ext or hi[il + 1:].max(initial=-np.inf) >= start:
            r["gate"] = "G2 IMPULSO H4 NON VALIDO (superato)"; return r
        rng = start - ext
        z50, z618 = ext + 0.5 * rng, ext + 0.618 * rng
    if rng < P["min_impulse_atr"] * a4:
        r["gate"] = "G2 IMPULSO H4 TROPPO PICCOLO"; return r
    zlo, zhi = min(z50, z618) - P["fib_tol_atr"] * a4, max(z50, z618) + P["fib_tol_atr"] * a4
    retr = (ext - price) / rng if d == "buy" else (price - ext) / rng
    r["Ritracciamento %"] = round(retr * 100, 1)
    if not (zlo <= price <= zhi):
        r["gate"] = f"G2 PREZZO FUORI ZONA 50-61.8% (ritracc. {retr*100:.0f}%)"; return r
    r["Fibonacci H4 zona"] = "50-61.8%"

    # G3 — H1: né in caduta libera né già ripartito (RSI H1 nella fascia)
    lo_r, hi_r = P["rsi_h1_long"] if d == "buy" else P["rsi_h1_short"]
    if not (lo_r <= r["RSI H1"] <= hi_r):
        r["gate"] = f"G3 RSI H1 FUORI FASCIA ({r['RSI H1']})"; return r

    # G4 — livello S/R H1 con >= 2 tocchi distinti, vicino al prezzo
    wh = h1.iloc[-P["level_lookback"]:]
    fh, fl = fractals(wh.high.values, wh.low.values)
    piv = wh.low.values[fl] if d == "buy" else wh.high.values[fh]
    tol = P["level_tol_atr"] * a1
    best, best_n = None, 0
    for p in piv:
        cl = piv[np.abs(piv - p) <= tol]
        near = abs(cl.mean() - price) <= 1.0 * a1 + tol
        side_ok = (cl.mean() <= price + tol) if d == "buy" else (cl.mean() >= price - tol)
        if near and side_ok and len(cl) > best_n:
            best, best_n = float(cl.mean()), len(cl)
    if best is None or best_n < P["min_touches"]:
        r["gate"] = "G4 NESSUN LIVELLO H1 CON 2 TOCCHI VICINO AL PREZZO"; return r
    r["Livello S/R H1"] = round(best, 6); r["Tocchi livello"] = best_n

    # G5 — pinbar M15 CHIUSA sul livello + RSI M15 in fascia e nella direzione giusta
    pin = None
    for j in range(1, P["pin_lookback"] + 1):
        b = m15.iloc[-j]
        o, h, l, c = b.open, b.high, b.low, b.close
        rg = h - l
        if rg <= 0 or rg < P["pin_min_atr"] * a15:
            continue
        body = abs(c - o)
        if d == "buy":
            wick = min(o, c) - l
            if wick >= P["pin_wick"] * rg and body <= P["pin_body"] * rg and l <= best + tol and c > best:
                pin = dict(idx=j, low=l, high=h, close=c); break
        else:
            wick = h - max(o, c)
            if wick >= P["pin_wick"] * rg and body <= P["pin_body"] * rg and h >= best - tol and c < best:
                pin = dict(idx=j, low=l, high=h, close=c); break
    if pin is None:
        r["gate"] = "G5 NESSUNA PINBAR M15 SUL LIVELLO"; return r
    rsi15, rsi15_prev = float(m15.rsi.iloc[-1]), float(m15.rsi.iloc[-2])
    lo_m, hi_m = P["rsi_m15_long"] if d == "buy" else P["rsi_m15_short"]
    rising = rsi15 > rsi15_prev if d == "buy" else rsi15 < rsi15_prev
    r["RSI M15"] = round(rsi15, 1)
    if not (lo_m <= rsi15 <= hi_m) or (pin["idx"] == 1 and not rising):
        r["gate"] = f"G5 RSI M15 NON CONFERMA ({rsi15:.0f})"; return r

    # G6 — stop, target, spazio fino al massimo/minimo precedente, spread
    entry = price
    if d == "buy":
        stop_struct = min(pin["low"], best) - P["stop_buffer_atr1"] * a1
        stop = min(stop_struct, entry - P["min_stop_atr4"] * a4)
        risk = entry - stop
        if (entry - pin["close"]) > P["max_chase_R"] * risk:
            r["gate"] = "G6 PREZZO GIÀ PARTITO (ingresso tardivo)"; return r
        tp = min(entry + P["rr_target"] * risk, ext - 0.1 * a4)
    else:
        stop_struct = max(pin["high"], best) + P["stop_buffer_atr1"] * a1
        stop = max(stop_struct, entry + P["min_stop_atr4"] * a4)
        risk = stop - entry
        if (pin["close"] - entry) > P["max_chase_R"] * risk:
            r["gate"] = "G6 PREZZO GIÀ PARTITO (ingresso tardivo)"; return r
        tp = max(entry - P["rr_target"] * risk, ext + 0.1 * a4)
    rr = abs(tp - entry) / risk if risk > 0 else 0
    r.update({"Entry": round(entry, 6), "SL": round(stop, 6), "TP": round(tp, 6),
              "R:R": round(rr, 2), "Stop %": round(risk / entry * 100, 3)})
    if rr < P["rr_min"]:
        r["gate"] = f"G6 SPAZIO INSUFFICIENTE PRIMA DEL LIVELLO H4 (R:R {rr:.2f})"; return r
    spread_R = SPREAD_PCT[cls] / 100 * entry / risk
    r["Spread/Rischio %"] = round(spread_R * 100, 1)
    if spread_R > P["max_spread_R"]:
        r["gate"] = f"G6 SPREAD TROPPO ALTO RISPETTO ALLO STOP ({spread_R*100:.0f}%)"; return r

    # BONUS (non obbligatori): 0–3
    div = rsi_divergence(h1, d)
    if div: r["bonus"] += 1; r["bonus_det"].append("B1 divergenza RSI H1")
    if abs(pin["close"] - float(m15.ema20.iloc[-pin["idx"]])) <= 0.5 * a15:
        r["bonus"] += 1; r["bonus_det"].append("B2 pinbar su EMA20 M15")
    if best_n >= 3:
        r["bonus"] += 1; r["bonus_det"].append("B3 livello H1 >= 3 tocchi")
    r["ok"] = True; r["gate"] = "TUTTI I GATE SUPERATI"
    r["Size"] = "piena" if r["bonus"] >= 2 else "50%"
    r["Rischio €"] = RISK_EUR if r["bonus"] >= 2 else RISK_EUR / 2
    r["Esposizione €"] = round(r["Rischio €"] / (risk / entry), 0)
    r["Setup descrizione"] = (f"{'Supporto' if d=='buy' else 'Resistenza'} H1 {best:.5g} "
                              f"({best_n} tocchi) in zona Fib 50-61.8% H4 + pinbar M15"
                              + ("" if not r["bonus_det"] else " + " + " + ".join(r["bonus_det"])))
    return r

def rsi_divergence(h1, d, window=30):
    w = h1.iloc[-window:]
    fh, fl = fractals(w.high.values, w.low.values)
    if d == "buy" and len(fl) >= 2:
        i1, i2 = fl[-2], fl[-1]
        return w.low.values[i2] < w.low.values[i1] and w.rsi.values[i2] > w.rsi.values[i1]
    if d == "sell" and len(fh) >= 2:
        i1, i2 = fh[-2], fh[-1]
        return w.high.values[i2] > w.high.values[i1] and w.rsi.values[i2] < w.rsi.values[i1]
    return False

# =============================================================================
# LOGICA v3 (riproduzione fedele, SOLO per il confronto nel backtest)
# =============================================================================
def _v3_ema(closes, period):
    arr = np.asarray(closes, dtype=float)
    if len(arr) < period: return None
    k = 2.0 / (period + 1); e = float(np.mean(arr[:period]))
    for p in arr[period:]: e = p * k + e * (1 - k)
    return e

def _v3_trend(df, n=8):
    lows = df.low.values[-n:]; highs = df.high.values[-n:]
    hl = sum(lows[i] > lows[i - 1] for i in range(1, len(lows)))
    lh = sum(highs[i] < highs[i - 1] for i in range(1, len(highs)))
    return "Rialzista" if hl >= 5 else "Ribassista" if lh >= 5 else "Laterale"

def evaluate_v3(h4, h1, m15, name, pin_lookback=1):
    """Stessa logica del notebook v3 (tail 250 H4 / 100 H1 / 30 M15), con score S1-S6 del codice v3."""
    h4 = h4.iloc[-250:]; h1 = h1.iloc[-100:]; m15 = m15.iloc[-30:]
    r = {"ok": False, "gate": ""}
    t4 = _v3_trend(h4); c4 = h4.close.values
    e50 = _v3_ema(c4, 50); e200 = _v3_ema(c4, 200)
    if e50 is None: r["gate"] = "dati"; return r
    p = c4[-1]
    if e200 is not None:
        al4 = "long" if p > e50 > e200 else "short" if p < e50 < e200 else "mista"
    else:
        al4 = "long" if p > e50 else "short"
    if t4 == "Laterale" and al4 == "mista": r["gate"] = "G2"; return r
    bias = t4 if t4 != "Laterale" else ("Rialzista" if "long" in al4 else "Ribassista")
    t1 = _v3_trend(h1)
    if (bias == "Rialzista" and t1 == "Ribassista") or (bias == "Ribassista" and t1 == "Rialzista") or t1 == "Laterale":
        r["gate"] = "G3"; return r
    trend = t1
    c1 = h1.close.values; e20_1 = _v3_ema(c1, 20); e50_1 = _v3_ema(c1, 50)
    al1 = "long" if c1[-1] > e20_1 > e50_1 else "short" if c1[-1] < e20_1 < e50_1 else "mista"
    piv = h1.tail(30).low.values if trend == "Rialzista" else h1.tail(30).high.values
    best, bn = None, 0
    for q in piv:
        cl = [x for x in piv if abs(x - q) / abs(q) <= 0.002]
        if len(cl) > bn: bn = len(cl); best = float(np.mean(cl))
    if best is None or bn < 2: r["gate"] = "G4"; return r
    e20_15 = _v3_ema(m15.close.values, 20)
    pin = None
    for j in range(1, pin_lookback + 1):
        b = m15.iloc[-j]; o, h, l, c = b.open, b.high, b.low, b.close; rg = h - l
        if rg == 0: continue
        near = abs((l if trend == "Rialzista" else h) - best) / abs(best) <= 0.003
        if not near: continue
        body = abs(c - o)
        if trend == "Rialzista" and (min(o, c) - l) >= 0.6 * rg and body <= 0.3 * rg:
            pin = dict(entry=c, low=l, high=h); break
        if trend == "Ribassista" and (h - max(o, c)) >= 0.6 * rg and body <= 0.3 * rg:
            pin = dict(entry=c, low=l, high=h); break
    if pin is None: r["gate"] = "G5"; return r
    rsi15 = float(rsi_wilder(m15.close).iloc[-1])
    rsi_ok = 25 <= rsi15 <= 75 and ((trend == "Rialzista" and rsi15 <= 70) or (trend == "Ribassista" and rsi15 >= 30))
    if trend == "Rialzista":
        sl = min(pin["low"], best) * 0.999; tp = pin["entry"] + 2.5 * (pin["entry"] - sl)
    else:
        sl = max(pin["high"], best) * 1.001; tp = pin["entry"] - 2.5 * (sl - pin["entry"])
    dirok = lambda a: ("long" in a and trend == "Rialzista") or ("short" in a and trend == "Ribassista")
    # BUG v3 riprodotto: tolleranza 5% -> la confluenza risulta sempre "38.2%" -> S3 mai vero
    score = int(dirok(al4)) + int(dirok(al1)) + 0 + 0 + int(rsi_ok)
    score += int(e20_15 is not None and abs(pin["entry"] - e20_15) / e20_15 <= 0.005)
    if score < 3: r["gate"] = "score<3"; return r
    r.update(ok=True, gate="v3 OK", Direzione="buy" if trend == "Rialzista" else "sell",
             Entry=pin["entry"], SL=sl, TP=tp, score=score)
    return r

# =============================================================================
# SCANNER LIVE
# =============================================================================
def run_scan(api_key, out_dir):
    P = PRESETS["v4"]
    now_utc = pd.Timestamp.now(tz="UTC"); now_rome = now_utc.tz_convert(ROME)
    log("=" * 64); log(f"  SCANNER H4+H1+M15 v4 — {now_rome:%Y-%m-%d %H:%M} (Roma)"); log("=" * 64)
    if weekend_block(now_rome):
        log("\n(!) Finestra weekend (ven 18:00 – dom): nessun nuovo ingresso. Analisi solo informativa.")
    open_pos = []
    op_path = os.path.join(out_dir, "posizioni_aperte.csv")
    if os.path.exists(op_path):
        try:
            op = pd.read_csv(op_path)
            open_pos = [(str(a).strip(), str(b).strip().lower()) for a, b in
                        zip(op["Nome prodotto"], op["Direzione"])]
            log(f"Posizioni aperte lette da posizioni_aperte.csv: {len(open_pos)}")
        except Exception as e:
            log(f"(!) posizioni_aperte.csv illeggibile: {e}")
    else:
        log("Suggerimento: crea posizioni_aperte.csv (colonne: Nome prodotto,Direzione) "
            "nella cartella TRADING_V4 per attivare il controllo esposizione.")

    rows = []
    names = [n for n in INSTRUMENTS]
    for i, name in enumerate(names, 1):
        sym, exch, cls, mkt = INSTRUMENTS[name]
        base = {"Nome prodotto": name, "Data e ora analisi": f"{now_rome:%Y-%m-%d %H:%M}",
                "Piattaforma dati": "Twelvedata"}
        log(f"  [{i:02d}/{len(names)}] {name:<24}")
        if not market_open(name, now_rome):
            rows.append({**base, "Esito": "MERCATO CHIUSO"}); continue
        dfs = {}
        err = None
        for tf, n in (("4h", 400), ("1h", 300), ("15min", 150)):
            df, msg = fetch_td(sym, exch, tf, n, api_key); time.sleep(REQUEST_DELAY)
            df = drop_incomplete(df, tf, now_utc)
            if not validate(df, {"4h": 230, "1h": 80, "15min": 40}[tf]):
                err = f"DATI {tf} NON VALIDI ({msg})"; break
            dfs[tf] = add_features(df, tf)
        if err:
            rows.append({**base, "Esito": err}); continue
        res = evaluate_v4(dfs["4h"], dfs["1h"], dfs["15min"], P, name)
        row = {**base, **{k: v for k, v in res.items() if k not in ("ok", "gate", "bonus_det")}}
        row["Esito"] = res["gate"]
        row["Bonus (0-3)"] = res.get("bonus", 0)
        if res["ok"]:
            row["Time stop (Roma)"] = f"{now_rome + timedelta(hours=P['time_stop_hours']):%Y-%m-%d %H:%M}"
        row["_ok"] = res["ok"]
        rows.append(row)
        log(f"        -> {row['Esito']}")

    # Filtri finali: weekend + esposizione (ordinati per bonus)
    sig = sorted([r for r in rows if r.get("_ok")], key=lambda r: -r["Bonus (0-3)"])
    accepted = list(open_pos)
    for r in sig:
        if weekend_block(now_rome):
            r["Esito"] = "SCARTATO: finestra weekend"; r["_ok"] = False; continue
        ok, why = exposure_ok(r["Nome prodotto"], r["Direzione"], accepted)
        if not ok:
            r["Esito"] = f"SCARTATO: {why}"; r["_ok"] = False; continue
        accepted.append((r["Nome prodotto"], r["Direzione"]))

    df = pd.DataFrame(rows)
    ts = now_rome.strftime("%Y%m%d_%H%M")
    path = os.path.join(out_dir, f"Scan_v4_{ts}.xlsx")
    export_scan(df, path)
    n_ok = int(df.get("_ok", pd.Series(dtype=bool)).fillna(False).sum())
    log(f"\nSegnali operativi: {n_ok}  |  File: {path}")
    if n_ok == 0:
        log("Nessun setup: con le regole v4 è NORMALE avere molti giorni senza segnali.")

SCAN_COLS = ["Nome prodotto", "Esito", "Direzione", "Entry", "SL", "TP", "R:R", "Stop %",
             "Size", "Rischio €", "Esposizione €", "Bonus (0-3)", "Setup descrizione",
             "Time stop (Roma)", "EMA H4", "Ritracciamento %", "Fibonacci H4 zona",
             "Livello S/R H1", "Tocchi livello", "RSI H1", "RSI M15", "ATR H4",
             "Spread/Rischio %", "Data e ora analisi", "Piattaforma dati"]

def export_scan(df, path):
    for c in SCAN_COLS:
        if c not in df.columns: df[c] = ""
    ok = df[df.get("_ok", False) == True][SCAN_COLS] if "_ok" in df else df.iloc[0:0][SCAN_COLS]
    legenda = pd.DataFrame({"Voce": [
        "G1", "G2", "G3", "G4", "G5", "G6", "Filtri finali", "B1-B3", "Size", "Esposizione €",
        "Time stop", "AVVERTENZA"], "Significato": [
        "Bias H4: EMA50>EMA200, EMA50 in salita, prezzo sopra EMA200 (long; speculare short)",
        "Impulso H4 >= 3 ATR e prezzo nella zona 50–61,8% (tolleranza 0,25 ATR H4)",
        "RSI H1 in 30–60 per long, 40–70 per short (né in caduta libera né già ripartito)",
        "Supporto/resistenza H1 con >= 2 tocchi distinti, entro ~1 ATR H1 dal prezzo",
        "Pinbar M15 CHIUSA sul livello, RSI M15 30–55 in risalita (long) / 45–70 in discesa (short)",
        "Stop >= 1 ATR H4; TP = 2,5R ma prima del massimo/minimo H4 precedente; R:R >= 2; spread <= 10% del rischio",
        "No nuovi ingressi ven 18:00–dom; max 1 posizione per valuta nella stessa direzione",
        "Bonus facoltativi: divergenza RSI H1, pinbar su EMA20 M15, livello con >= 3 tocchi",
        "Bonus >= 2: rischio pieno; altrimenti metà rischio",
        "Controvalore della posizione (NON il margine) per perdere esattamente 'Rischio €' allo stop",
        "Se dopo 10 giorni né TP né SL: chiudi a mercato",
        "Parametri NON ancora validati: esegui prima MODE='backtest'"]})
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        ok.to_excel(xw, sheet_name="Segnali operativi", index=False)
        df[SCAN_COLS].to_excel(xw, sheet_name="Tutti gli strumenti", index=False)
        legenda.to_excel(xw, sheet_name="Legenda", index=False)
        for ws in xw.book.worksheets:
            ws.freeze_panes = "A2"
            for col in ws.columns:
                ws.column_dimensions[col[0].column_letter].width = min(
                    60, max(10, max(len(str(c.value or "")) for c in col[:50]) + 2))

# =============================================================================
# BACKTEST
# =============================================================================
def load_history(name, api_key, cache_dir):
    """Scarica 5000 candele per timeframe e le UNISCE alla cache: la storia cresce a ogni esecuzione."""
    sym, exch, _, _ = INSTRUMENTS[name]
    out = {}
    for tf in ("4h", "1h", "15min"):
        fn = os.path.join(cache_dir, f"{re.sub(r'[^A-Za-z0-9]', '_', name)}_{tf}.csv")
        old = None
        if os.path.exists(fn):
            old = pd.read_csv(fn); old["time"] = pd.to_datetime(old["time"], utc=True)
        new, msg = (None, "offline") if not api_key else fetch_td(sym, exch, tf, 5000, api_key)
        if api_key: time.sleep(REQUEST_DELAY)
        df = pd.concat([d for d in (old, new) if d is not None], ignore_index=True) if (old is not None or new is not None) else None
        if df is None:
            log(f"    {name} {tf}: nessun dato ({msg})"); return None
        df = df.sort_values("time").drop_duplicates("time", keep="last").reset_index(drop=True)
        df.to_csv(fn, index=False)
        out[tf] = df
    return out

def simulate_trade(m15, i, sig, P, cls):
    """Segue il trade candela per candela dalla M15 successiva. Esito in R, costi inclusi."""
    d = 1 if sig["Direzione"] == "buy" else -1
    entry, sl, tp = sig["Entry"], sig["SL"], sig["TP"]
    risk = abs(entry - sl)
    cost_R = SPREAD_PCT[cls] / 100 * entry / risk
    t0 = m15.time.iloc[i] + pd.Timedelta(minutes=15)
    deadline = t0 + pd.Timedelta(hours=P.get("time_stop_hours", 10 ** 6)) if P else None
    H, L, O, C, T = (m15.high.values, m15.low.values, m15.open.values, m15.close.values, m15.time.values)
    for j in range(i + 1, len(m15)):
        # gap all'apertura oltre lo stop (es. riapertura domenica): esce al prezzo di apertura
        if d == 1:
            if L[j] <= sl:
                px = min(O[j], sl); return j, (px - entry) / risk - cost_R, "SL"
            if H[j] >= tp: return j, (tp - entry) / risk - cost_R, "TP"
        else:
            if H[j] >= sl:
                px = max(O[j], sl); return j, (entry - px) / risk - cost_R, "SL"
            if L[j] <= tp: return j, (entry - tp) / risk - cost_R, "TP"
        tj = pd.Timestamp(T[j]).tz_localize("UTC") if pd.Timestamp(T[j]).tzinfo is None else pd.Timestamp(T[j])
        tclose = tj + pd.Timedelta(minutes=15)
        if deadline is not None and tclose >= deadline:
            return j, d * (C[j] - entry) / risk - cost_R, "TIME"
        if P and P.get("close_before_weekend"):
            tr = tclose.tz_convert(ROME)
            if tr.weekday() == 4 and tr.hour >= 21:
                return j, d * (C[j] - entry) / risk - cost_R, "WEEKEND"
    return len(m15) - 1, d * (C[-1] - entry) / risk - cost_R, "APERTO A FINE DATI"

GATE_COUNT = {}   # preset -> {motivo: n. candele M15 scartate}

def backtest_symbol(name, data, preset):
    P = PRESETS.get(preset)
    gc = GATE_COUNT.setdefault(preset, {})
    cls = INSTRUMENTS[name][2]
    h4, h1, m15 = data["4h"], data["1h"], data["15min"]
    if P:
        h4, h1, m15 = add_features(h4, "4h"), add_features(h1, "1h"), add_features(m15, "15min")
        P = dict(P, pin_lookback=1)
    c4 = (h4.time + pd.Timedelta(hours=4)).values
    c1 = (h1.time + pd.Timedelta(hours=1)).values
    c15 = (m15.time + pd.Timedelta(minutes=15)).values
    trades, busy_until = [], -1
    for i in range(60, len(m15)):
        if i <= busy_until: continue
        t = c15[i]
        k4 = np.searchsorted(c4, t, side="right") - 1
        k1 = np.searchsorted(c1, t, side="right") - 1
        if k4 < 230 or k1 < 80: continue
        tr = pd.Timestamp(t).tz_localize("UTC").tz_convert(ROME) if pd.Timestamp(t).tzinfo is None \
            else pd.Timestamp(t).tz_convert(ROME)
        if P and P["weekend_filter"] and weekend_block(tr): continue
        H4, H1, M15 = h4.iloc[:k4 + 1], h1.iloc[:k1 + 1], m15.iloc[:i + 1]
        sig = evaluate_v4(H4, H1, M15, P, name) if P else evaluate_v3(H4, H1, M15, name)
        if not sig["ok"]:
            g = sig["gate"].split(" (")[0]; gc[g] = gc.get(g, 0) + 1
            continue
        if abs(sig["Entry"] - sig["SL"]) <= 0: continue
        j, R, why = simulate_trade(m15, i, sig, P, cls)
        busy_until = j
        trades.append({"Preset": preset, "Nome prodotto": name, "Direzione": sig["Direzione"],
                       "Ingresso (Roma)": tr.strftime("%Y-%m-%d %H:%M"),
                       "Uscita (Roma)": pd.Timestamp(c15[j]).tz_localize("UTC").tz_convert(ROME).strftime("%Y-%m-%d %H:%M")
                       if pd.Timestamp(c15[j]).tzinfo is None else pd.Timestamp(c15[j]).tz_convert(ROME).strftime("%Y-%m-%d %H:%M"),
                       "_t_in": pd.Timestamp(t), "_t_out": pd.Timestamp(c15[j]),
                       "Entry": sig["Entry"], "SL": sig["SL"], "TP": sig["TP"],
                       "Stop %": round(abs(sig["Entry"] - sig["SL"]) / sig["Entry"] * 100, 3),
                       "Bonus": sig.get("bonus", sig.get("score")), "Uscita per": why, "R": round(R, 3)})
    return trades

def apply_exposure(trades, enabled):
    """Scarta, in ordine cronologico, i trade che raddoppierebbero l'esposizione su una valuta."""
    if not enabled: return trades
    kept, openl = [], []
    for t in sorted(trades, key=lambda x: x["_t_in"]):
        openl = [o for o in openl if o["_t_out"] > t["_t_in"]]
        ok, _ = exposure_ok(t["Nome prodotto"], t["Direzione"], [(o["Nome prodotto"], o["Direzione"]) for o in openl])
        if ok:
            kept.append(t); openl.append(t)
    return kept

def stats(tr, months):
    if not tr:
        return {"Trade": 0}
    R = pd.Series([t["R"] for t in sorted(tr, key=lambda x: x["_t_out"])])
    eq = R.cumsum(); dd = (eq - eq.cummax()).min()
    streak = mx = 0
    for x in R:
        streak = streak + 1 if x < 0 else 0; mx = max(mx, streak)
    wins, loss = R[R > 0].sum(), -R[R < 0].sum()
    return {"Trade": len(R), "Trade/mese": round(len(R) / max(months, 0.1), 1),
            "Vincenti %": round((R > 0).mean() * 100, 1), "R medio (expectancy)": round(R.mean(), 3),
            "R totale": round(R.sum(), 2), "Profit factor": round(wins / loss, 2) if loss > 0 else None,
            "Max drawdown (R)": round(dd, 2), "Max perdite consecutive": mx,
            "Usciti per time stop %": round(np.mean([t["Uscita per"] == "TIME" for t in tr]) * 100, 1)}

def run_backtest(api_key, out_dir):
    log("=" * 64); log("  BACKTEST v3 vs v4 — H4+H1+M15"); log("=" * 64)
    cache = os.path.join(out_dir, "cache")
    data = {}
    for n in BACKTEST_SYMBOLS:
        log(f"  Dati {n} ...")
        d = load_history(n, api_key, cache)
        if d and all(validate(d[tf], 300) for tf in d):
            data[n] = d
    if not data:
        log("Nessun dato disponibile."); return
    first = max(d["15min"].time.iloc[0] for d in data.values())
    last = min(d["15min"].time.iloc[-1] for d in data.values())
    months = (last - first).days / 30.4
    log(f"\nPeriodo coperto dalle M15: {first:%Y-%m-%d} -> {last:%Y-%m-%d} (~{months:.1f} mesi)")
    all_tr, summary = [], []
    for preset in BACKTEST_PRESETS:
        tr = []
        for n, d in data.items():
            log(f"  [{preset}] {n}"); tr += backtest_symbol(n, d, preset)
        tr = apply_exposure(tr, PRESETS.get(preset, {}).get("exposure_filter", False))
        st = stats(tr, months); st["Preset"] = preset; summary.append(st); all_tr += tr
    sm = pd.DataFrame(summary).set_index("Preset")
    log("\n" + sm.to_string())
    df = pd.DataFrame(all_tr).drop(columns=["_t_in", "_t_out"], errors="ignore")
    by_sym = (df.groupby(["Preset", "Nome prodotto"]).R.agg(["count", "mean", "sum"]).round(3)
              .rename(columns={"count": "Trade", "mean": "R medio", "sum": "R totale"})) if not df.empty else pd.DataFrame()
    note = pd.DataFrame({"Come leggere": [
        "R = multipli del rischio. +1R = hai guadagnato quanto rischiavi; -1R = stop pieno.",
        "Expectancy (R medio) > 0 dopo i costi = la strategia ha un vantaggio SU QUESTO CAMPIONE.",
        "Servono almeno 100 trade per preset prima di trarre conclusioni; sotto i 30 è rumore.",
        "La cache in TRADING_V4/cache si allunga a ogni esecuzione: rilancia il backtest ogni 2-4 settimane.",
        "Spread = stime (SPREAD_PCT). Nessuno slippage oltre ai gap. Candela con SL e TP insieme = SL (prudente).",
        "Motivi di scarto: mostra quale regola blocca più spesso i segnali (utile se i trade sono pochissimi).",
        "Il preset v3 riproduce la logica precedente, inclusi i suoi bug, per avere un confronto onesto.",
        "Risultati IN-SAMPLE: se modifichi i parametri per migliorare questi numeri rischi l'overfitting."]})
    ts = datetime.now().strftime("%Y%m%d_%H%M")
    path = os.path.join(out_dir, f"Backtest_v3_vs_v4_{ts}.xlsx")
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        sm.to_excel(xw, sheet_name="Riepilogo")
        if not by_sym.empty: by_sym.to_excel(xw, sheet_name="Per strumento")
        df.to_excel(xw, sheet_name="Trade", index=False)
        gates = pd.DataFrame([(p, g, n) for p, d in GATE_COUNT.items() for g, n in d.items()],
                             columns=["Preset", "Motivo di scarto", "Candele M15"])
        if not gates.empty:
            gates.sort_values(["Preset", "Candele M15"], ascending=[True, False]).to_excel(
                xw, sheet_name="Motivi di scarto", index=False)
        note.to_excel(xw, sheet_name="Come leggere", index=False)
    log(f"\nFile: {path}")
    log("NOTA: risultati su un campione breve e in-sample. Non sono una previsione.")

# =============================================================================
def main():
    api_key = get_api_key()
    out_dir = get_out_dir()
    if not api_key:
        if MODE == "scan":
            raise SystemExit("API key Twelvedata assente: imposta la variabile TWELVEDATA_API_KEY "
                             "(o i Colab Secrets).")
        log("(!) API key assente: il backtest userà solo i dati già presenti in cache.")
    run_scan(api_key, out_dir) if MODE == "scan" else run_backtest(api_key, out_dir)

if __name__ == "__main__":
    main()

# =============================================================================
# CHANGELOG v3 -> v4
# -----------------------------------------------------------------------------
# BUG CORRETTI
# - find_fib_zone / check_fib_confluence: tol_pct=0.05 (5% del prezzo). Su EUR/USD
#   il 5% sono ~570 pip: il primo livello testato (38.2) era sempre "vicino".
#   Effetto: zona sempre 38.2%, bonus Fib 50/61.8 (S3) mai assegnato.
# - Candela in formazione usata come chiusa (H4, H1 e soprattutto M15/pinbar).
# - Swing Fibonacci: prendeva ultimo swing high e ultimo swing low senza verificare
#   l'ordine (minimo PRIMA del massimo) né che l'impulso fosse ancora intatto.
# - "Rimbalzi ult. 8 H1" contava candele con minimi crescenti, non tocchi di un livello.
# - Etichette S1-S6 diverse tra prompt (S1 trend, S5 divergenza...) e codice
#   (S1 EMA H4, S5 RSI M15...). Il journal risultava illeggibile.
# - RSI M15: il codice accettava 25-70 (long) contro il 30-50 in risalita del prompt.
# - Forex considerato aperto anche il sabato/domenica.
# - Chiamata RSI separata a Twelvedata (spreco di 1 chiamata su 4): ora calcolato in locale.
# SCELTE DI STRATEGIA (ipotesi da validare col backtest)
# - Stop minimo 1 ATR H4 e size a rischio fisso; TP limitato dal massimo/minimo H4.
# - Zona 50-61.8% obbligatoria; H1 filtrato con RSI 30-60 invece di "prezzo > EMA20 H1"
#   (la vecchia regola, durante un ritracciamento profondo, era quasi sempre falsa:
#   per questo la v3 entrava solo su ritracciamenti minimi e con RSI alto).
# - Filtro weekend, filtro esposizione per valuta, time stop 10 giorni.
# - Ensemble multi-sorgente rimosso: i suoi componenti (R:R fisso, "rimbalzi",
#   Fib sempre 38.2) non aggiungevano informazione; yfinance H4 ricampionato escluso.
# =============================================================================
